In [ ]:
import pandas as pd
import joblib
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

# Load the dataset
df = pd.read_csv(r"C:\Users\Rithik\Downloads\gym_members_exercise_tracking3.csv")

# Drop any duplicate or unnecessary columns
df = df.drop(columns=['Session_Duration (mins)'])  # Keep 'Session_Duration (minutes)'

# Separate features and target variable
X = df.drop(columns=['Calories_Burned'])
y = df['Calories_Burned']

# Encode categorical variables
label_encoders = {}
categorical_columns = ['Gender', 'Workout_Type', 'Diet_Type']
for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le  # Save the encoders for later use

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define Optuna optimization function
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0)
    }

    model = xgb.XGBRegressor(**params, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return mean_squared_error(y_test, y_pred)  # Minimize MSE

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

# Best hyperparameters
best_params = study.best_params
print("Best Parameters:", best_params)

# Train the final XGBoost model with best hyperparameters
best_model = xgb.XGBRegressor(**best_params, random_state=42)
best_model.fit(X_train, y_train)

# Evaluate the model
train_score = best_model.score(X_train, y_train)
test_score = best_model.score(X_test, y_test)
print(f"Optimized XGBoost Training R^2 Score: {train_score:.4f}")
print(f"Optimized XGBoost Testing R^2 Score: {test_score:.4f}")

# Save the trained model and encoders
joblib.dump(best_model, r"C:\Users\Rithik\Downloads\xgb_calories_model.joblib")
joblib.dump(label_encoders, r"C:\Users\Rithik\Downloads\label_encoders.joblib")

print("Optimized XGBoost Model and label encoders saved successfully!")


[I 2025-04-02 13:41:34,523] A new study created in memory with name: no-name-b909c0ac-6e69-4bd3-93d2-54a7d72963b4
[I 2025-04-02 13:41:34,978] Trial 0 finished with value: 1121.9803466796875 and parameters: {'n_estimators': 300, 'learning_rate': 0.02917766641613058, 'max_depth': 6, 'subsample': 0.7188032012361134, 'colsample_bytree': 0.9215093184243934, 'reg_alpha': 0.3022892106151315, 'reg_lambda': 0.6674254811823047}. Best is trial 0 with value: 1121.9803466796875.
[I 2025-04-02 13:41:35,558] Trial 1 finished with value: 1804.450927734375 and parameters: {'n_estimators': 500, 'learning_rate': 0.012810013148407969, 'max_depth': 8, 'subsample': 0.6700603476820374, 'colsample_bytree': 0.8553572774483866, 'reg_alpha': 0.8584044984145939, 'reg_lambda': 0.8367829681084974}. Best is trial 0 with value: 1121.9803466796875.
[I 2025-04-02 13:41:35,961] Trial 2 finished with value: 5852.541015625 and parameters: {'n_estimators': 500, 'learning_rate': 0.1679264053187398, 'max_depth': 9, 'subsampl

Best Parameters: {'n_estimators': 200, 'learning_rate': 0.08383559651229099, 'max_depth': 4, 'subsample': 0.9230641116176058, 'colsample_bytree': 0.8049987865922269, 'reg_alpha': 0.20462535330061787, 'reg_lambda': 0.3210834295193818}
Optimized XGBoost Training R^2 Score: 0.9996
Optimized XGBoost Testing R^2 Score: 0.9932
Optimized XGBoost Model and label encoders saved successfully!
